## ÉTAPE 9 — Corrélation et Heatmap
Objectif : analyser la relation entre
surface cultivée (Ha) et rendement (Hg/Ha)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


## 1. CHARGER ET PRÉPARER LES DONNÉES


In [ ]:
df = pd.read_csv("JEU_DE_DONNEE.csv")

# On garde les données mondiales ("World +") pour éviter
# la confusion entre pays — on analyse la tendance globale
df_monde = df[
    (df["country_or_area"] == "World +") &
    (df["element"].isin(["Area Harvested", "Yield", "Production Quantity"]))
].dropna(subset=["value"])

print(f"Lignes disponibles : {len(df_monde):,}")
print(f"Cultures disponibles : {df_monde['category'].nunique()}")


## 2. RESTRUCTURER — TABLEAU AVEC 3 COLONNES


## On veut un tableau avec :
  - une colonne "Area Harvested" (surface en Ha)
  - une colonne "Yield"          (rendement en Hg/Ha)
  - une colonne "Production"     (quantité en tonnes)
Pour pouvoir calculer les corrélations entre elles


In [ ]:
pivot = df_monde.pivot_table(
    index=["category", "year"],
    columns="element",
    values="value",
    aggfunc="sum"
).reset_index()

# Renommer pour simplifier
pivot = pivot.rename(columns={
    "Area Harvested"     : "surface_ha",
    "Yield"              : "rendement_hg_ha",
    "Production Quantity": "production_tonnes"
})

# Supprimer les lignes incomplètes
pivot = pivot.dropna(subset=["surface_ha", "rendement_hg_ha"])

print(f"\nTableau restructuré : {pivot.shape}")
print(pivot.head(5).to_string())


## 3. CALCULER LA CORRÉLATION GLOBALE


## La corrélation mesure si deux variables évoluent ensemble
+1 = parfaitement liées (quand l'une monte, l'autre aussi)
 0 = aucune relation
-1 = inverses (quand l'une monte, l'autre descend)


In [ ]:
corr_global = pivot[["surface_ha", "rendement_hg_ha", "production_tonnes"]].corr()

print("\n--- Matrice de corrélation globale ---")
print(corr_global.round(3))


## 4. GRAPHIQUE 1 — HEATMAP DE CORRÉLATION


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

labels = ["Surface (Ha)", "Rendement (Hg/Ha)", "Production (t)"]
data   = corr_global.values

# Dessiner la heatmap manuellement avec imshow
im = ax.imshow(data, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.8, label="Coefficient de corrélation")

# Annoter chaque case avec la valeur
for i in range(len(labels)):
    for j in range(len(labels)):
        val = data[i, j]
        couleur_texte = "white" if abs(val) > 0.7 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=13, fontweight="bold", color=couleur_texte)

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_yticklabels(labels, fontsize=10)
ax.set_title("Heatmap de corrélation\n(Surface × Rendement × Production)",
             fontsize=13, fontweight="bold", pad=12)

plt.tight_layout()
plt.savefig("heatmap_correlation.png", dpi=150, bbox_inches="tight")
print("\nHeatmap sauvegardée ✓")
plt.show()


## 5. GRAPHIQUE 2 — SCATTER : SURFACE vs RENDEMENT


## Chaque point = une culture × une année


In [ ]:
fig2, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Scatter 1 : Surface vs Rendement ---
ax1 = axes[0]
sc1 = ax1.scatter(
    pivot["surface_ha"] / 1_000_000,   # en millions d'hectares
    pivot["rendement_hg_ha"] / 10_000, # en tonnes/Ha (plus lisible)
    alpha=0.3, s=20, color="#2563EB"
)

# Ligne de tendance
z = np.polyfit(
    pivot["surface_ha"].dropna(),
    pivot["rendement_hg_ha"].dropna(), 1
)
p = np.poly1d(z)
x_range = np.linspace(pivot["surface_ha"].min(), pivot["surface_ha"].max(), 100)
ax1.plot(x_range / 1_000_000, p(x_range) / 10_000,
         color="red", linewidth=2, linestyle="--", label="Tendance")

# Corrélation
corr_val = pivot["surface_ha"].corr(pivot["rendement_hg_ha"])
ax1.set_title(f"Surface vs Rendement\n(corrélation = {corr_val:.2f})",
              fontsize=12, fontweight="bold")
ax1.set_xlabel("Surface cultivée (millions d'hectares)")
ax1.set_ylabel("Rendement (tonnes / Ha)")
ax1.legend()
ax1.grid(alpha=0.3)

# --- Scatter 2 : Surface vs Production ---
ax2 = axes[1]
ax2.scatter(
    pivot["surface_ha"] / 1_000_000,
    pivot["production_tonnes"] / 1_000_000,
    alpha=0.3, s=20, color="#16A34A"
)

z2 = np.polyfit(
    pivot["surface_ha"].dropna(),
    pivot["production_tonnes"].dropna(), 1
)
p2 = np.poly1d(z2)
ax2.plot(x_range / 1_000_000, p2(x_range) / 1_000_000,
         color="red", linewidth=2, linestyle="--", label="Tendance")

corr_val2 = pivot["surface_ha"].corr(pivot["production_tonnes"])
ax2.set_title(f"Surface vs Production\n(corrélation = {corr_val2:.2f})",
              fontsize=12, fontweight="bold")
ax2.set_xlabel("Surface cultivée (millions d'hectares)")
ax2.set_ylabel("Production (millions de tonnes)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle("Relations entre surface, rendement et production",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("scatter_correlation.png", dpi=150, bbox_inches="tight")
print("Scatter plots sauvegardés ✓")
plt.show()


## 6. ZOOM SUR 5 CULTURES — ÉVOLUTION DANS LE TEMPS


## Est-ce que le rendement augmente même quand la surface stagne ?


In [ ]:
cultures_zoom = ["wheat", "maize", "rice_paddy", "sugar_cane", "roots_and_tubers_total"]
couleurs_zoom  = ["#2563EB", "#DC2626", "#16A34A", "#D97706", "#7C3AED"]

fig3, axes3 = plt.subplots(1, 2, figsize=(15, 6))

for culture, couleur in zip(cultures_zoom, couleurs_zoom):
    data_c = pivot[pivot["category"] == culture].sort_values("year")
    if len(data_c) < 5:
        continue

    # Graphique surface
    axes3[0].plot(data_c["year"], data_c["surface_ha"] / 1_000_000,
                  label=culture, color=couleur, linewidth=2, marker="o", markersize=3)

    # Graphique rendement
    axes3[1].plot(data_c["year"], data_c["rendement_hg_ha"] / 10_000,
                  label=culture, color=couleur, linewidth=2, marker="o", markersize=3)

axes3[0].set_title("Évolution de la surface cultivée", fontsize=12, fontweight="bold")
axes3[0].set_xlabel("Année")
axes3[0].set_ylabel("Surface (millions d'hectares)")
axes3[0].legend(fontsize=9)
axes3[0].grid(alpha=0.3)

axes3[1].set_title("Évolution du rendement", fontsize=12, fontweight="bold")
axes3[1].set_xlabel("Année")
axes3[1].set_ylabel("Rendement (tonnes / Ha)")
axes3[1].legend(fontsize=9)
axes3[1].grid(alpha=0.3)

plt.suptitle("Surface vs Rendement par culture (1961–2007)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("evolution_surface_rendement.png", dpi=150, bbox_inches="tight")
print("Graphique évolution sauvegardé ✓")
plt.show()


## 7. TABLEAU RÉCAPITULATIF — CORRÉLATION PAR CULTURE


In [ ]:
print("\n--- Corrélation Surface vs Rendement par culture (Top 15) ---")
correlations = []

for culture in pivot["category"].unique():
    data_c = pivot[pivot["category"] == culture].dropna()
    if len(data_c) < 10:
        continue
    corr = data_c["surface_ha"].corr(data_c["rendement_hg_ha"])
    correlations.append({"Culture": culture, "Corrélation": round(corr, 3)})

df_corr = (
    pd.DataFrame(correlations)
    .sort_values("Corrélation", ascending=False)
    .reset_index(drop=True)
)
df_corr.index = df_corr.index + 1
df_corr.index.name = "Rang"

print(df_corr.head(15).to_string())
print("\n→ Corrélation proche de +1 : plus de surface = meilleur rendement")
print("→ Corrélation proche de  0 : surface et rendement indépendants")
print("→ Corrélation proche de -1 : plus de surface = rendement qui baisse")
